In [ ]:
!pip install flash-attn -q

In [ ]:
import math
import re
from random import *
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# native

In [ ]:
# 训练词和句掩码策略
# sample IsNext and NotNext to be same in small batch size
def make_batch():
    batch = []
    positive = negative = 0       # 为了记录NSP任务中正样本和负样本的个数，比例最好是在一个batch中接近1:1
    while positive != batch_size/2 or negative != batch_size/2:
        tokens_a_index, tokens_b_index = randrange(len(sentences)), randrange(len(sentences))   # sample random index in sentences
        tokens_a, tokens_b = token_list[tokens_a_index], token_list[tokens_b_index]
        input_ids = [word_dict['[CLS]']] + tokens_a + [word_dict['[SEP]']] + tokens_b + [word_dict['[SEP]']]
        segment_ids = [0] * (1 + len(tokens_a) + 1) + [1] * (len(tokens_b) + 1)

        # MASK LM
        n_pred =  min(max_pred, max(1, int(round(len(input_ids) * 0.15))))   # 15 % of tokens in one sentence
        cand_maked_pos = [i for i, token in enumerate(input_ids)
                          if token != word_dict['[CLS]'] and token != word_dict['[SEP]']]
        shuffle(cand_maked_pos)
        masked_tokens, masked_pos = [], []
        for pos in cand_maked_pos[:n_pred]:
            masked_pos.append(pos)
            masked_tokens.append(input_ids[pos])
            if random() < 0.8:  # 80%
                input_ids[pos] = word_dict['[MASK]']   # make mask
            elif random() < 0.5:  # 10%
                index = randint(0, vocab_size - 1)   # random index in vocabulary
                input_ids[pos] = word_dict[number_dict[index]] # replace

        # Zero Paddings
        n_pad = maxlen - len(input_ids)
        input_ids.extend([0] * n_pad)
        segment_ids.extend([0] * n_pad)

        # Zero Padding (100% - 15%) tokens
        if max_pred > n_pred:
            n_pad = max_pred - n_pred
            masked_tokens.extend([0] * n_pad)
            masked_pos.extend([0] * n_pad)

        if tokens_a_index + 1 == tokens_b_index and positive < batch_size/2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, 1]) # IsNext
            positive += 1
        elif tokens_a_index + 1 != tokens_b_index and negative < batch_size/2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, 0]) # NotNext
            negative += 1
    return batch
# Proprecessing Finished

In [ ]:

def get_attn_pad_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    batch_size, len_k = seq_k.size()
    # eq(zero) is PAD token
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1)  # batch_size x 1 x len_k(=len_q), one is masking
    return pad_attn_mask.expand(batch_size, len_q, len_k)  # batch_size x len_q x len_k

In [ ]:
# 向量化
class Embedding(nn.Module):
    def __init__(self):
        super(Embedding, self).__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)  # token embedding
        self.pos_embed = nn.Embedding(maxlen, d_model)  # position embedding
        self.seg_embed = nn.Embedding(n_segments, d_model)  # segment(token type) embedding
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, seg):
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long)
        pos = pos.unsqueeze(0).expand_as(x)  # (seq_len,) -> (batch_size, seq_len)
        embedding = self.tok_embed(x) + self.pos_embed(pos) + self.seg_embed(seg)
        return self.norm(embedding)

In [ ]:

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(d_k) # scores : [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        scores.masked_fill_(attn_mask, -1e9) # Fills elements of self tensor with value where mask is one.
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn

In [ ]:
# 多头注意力

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super(MultiHeadAttention, self).__init__()
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)

    def forward(self, Q, K, V, attn_mask):
        # q: [batch_size x len_q x d_model], k: [batch_size x len_k x d_model], v: [batch_size x len_k x d_model]
        residual, batch_size = Q, Q.size(0)

        # (B, S, D) -proj-> (B, S, D) -split-> (B, S, H, W) -trans-> (B, H, S, W)
        q_s = self.W_Q(Q).view(batch_size, -1, n_heads, d_k).transpose(1,2)  # q_s: [batch_size x n_heads x len_q x d_k]
        k_s = self.W_K(K).view(batch_size, -1, n_heads, d_k).transpose(1,2)  # k_s: [batch_size x n_heads x len_k x d_k]
        v_s = self.W_V(V).view(batch_size, -1, n_heads, d_v).transpose(1,2)  # v_s: [batch_size x n_heads x len_k x d_v]

        attn_mask = attn_mask.unsqueeze(1).repeat(1, n_heads, 1, 1) # attn_mask : [batch_size x n_heads x len_q x len_k]

        # context: [batch_size x n_heads x len_q x d_v], attn: [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        context, attn = ScaledDotProductAttention()(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, n_heads * d_v) # context: [batch_size x len_q x n_heads * d_v]
        output = nn.Linear(n_heads * d_v, d_model)(context)
        return nn.LayerNorm(d_model)(output + residual), attn # output: [batch_size x len_q x d_model]

In [ ]:
# 位置编码

# def gelu(x):
#     "Implementation of the gelu activation function by Hugging Face"
#     return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))  

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU()

    def forward(self, x):
        # (batch_size, len_seq, d_model) -> (batch_size, len_seq, d_ff) -> (batch_size, len_seq, d_model)
        return self.fc2(self.activation(self.fc1(x)))

In [ ]:
# 编码器
class EncoderLayer(nn.Module):
    def __init__(self):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention()
        self.pos_ffn = PoswiseFeedForwardNet()

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask) # enc_inputs to same Q,K,V
        enc_outputs = self.pos_ffn(enc_outputs) # enc_outputs: [batch_size x len_q x d_model]
        return enc_outputs, attn

## 网络构建

In [ ]:
class BERT(nn.Module):
    def __init__(self):
        super(BERT, self).__init__()
        self.embedding = Embedding()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, d_model)
        self.activ1 = nn.Tanh()
        self.linear = nn.Linear(d_model, d_model)
        self.activ2 = nn.GELU()  # self.activ2 = gelu
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)
        # decoder is shared with embedding layer
        embed_weight = self.embedding.tok_embed.weight
        n_vocab, n_dim = embed_weight.size()
        self.decoder = nn.Linear(n_dim, n_vocab, bias=False)
        self.decoder.weight = embed_weight
        self.decoder_bias = nn.Parameter(torch.zeros(n_vocab))

    def forward(self, input_ids, segment_ids, masked_pos):
        output = self.embedding(input_ids, segment_ids)
        enc_self_attn_mask = get_attn_pad_mask(input_ids, input_ids)
        for layer in self.layers:
            output, enc_self_attn = layer(output, enc_self_attn_mask)
        # output : [batch_size, len, d_model], attn : [batch_size, n_heads, d_mode, d_model]
        # it will be decided by first token(CLS)
        h_pooled = self.activ1(self.fc(output[:, 0])) # [batch_size, d_model]
        logits_clsf = self.classifier(h_pooled) # [batch_size, 2]

        masked_pos = masked_pos[:, :, None].expand(-1, -1, output.size(-1)) # [batch_size, max_pred, d_model]
        # get masked position from final output of transformer.
        h_masked = torch.gather(output, 1, masked_pos) # masking position [batch_size, max_pred, d_model]
        h_masked = self.norm(self.activ2(self.linear(h_masked)))
        logits_lm = self.decoder(h_masked) + self.decoder_bias # [batch_size, max_pred, n_vocab]

        return logits_lm, logits_clsf

In [ ]:
# BERT Parameters
maxlen = 30  # maximum of length
batch_size = 6
max_pred = 5  # max tokens of prediction
n_layers = 6  # number of Encoder of Encoder Layer
n_heads = 12  # number of heads in Multi-Head Attention
d_model = 768  # Embedding Size
d_ff = 768 * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

## 数据准备

In [ ]:
text = (
    'Hello, I am Romeo. How are you?\n'
    'Fine, Romeo. My name is Juliet. Nice to meet you.\n'
    'Nice to meet you too. How is everything today?\n'
    'Great. My baseball team just won a game.\n'
    'Waah! Congratulations, Juliet!\n'
    'Thank you, Romeo.'
)
text

In [ ]:


sentences = re.sub("[.,!?\\-]", '', text.lower()).split('\n')  # filter '.', ',', '?', '!'
word_list = list(set(" ".join(sentences).split()))
word_dict = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}

for i, w in enumerate(word_list):
    word_dict[w] = i + 4

number_dict = {i: w for i, w in enumerate(word_dict)}
vocab_size = len(word_dict)
token_list = list()

for sentence in sentences:
    arr = [word_dict[s] for s in sentence.split()]
    token_list.append(arr)

In [ ]:
print("字典数量：", vocab_size)
print("原始句子的表示：", token_list)

## 模型训练

In [ ]:
# BERT Parameters
maxlen = 30  # maximum of length
batch_size = 6
max_pred = 5  # max tokens of prediction
n_layers = 6  # number of Encoder of Encoder Layer
n_heads = 12  # number of heads in Multi-Head Attention
d_model = 768  # Embedding Size
d_ff = 768 * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = BERT().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model

In [ ]:
batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))

for epoch in range(100):
    optimizer.zero_grad()
    logits_lm, logits_clsf = model(input_ids, segment_ids, masked_pos)
    loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens)     # for masked LM
    loss_lm = (loss_lm.float()).mean()
    loss_clsf = criterion(logits_clsf, isNext)     # for sentence classification
    loss = loss_lm + loss_clsf

    if (epoch + 1) % 10 == 0:
        print('Epoch:', '%04d' % (epoch + 1), 'cost =', '{:.6f}'.format(loss))

    loss.backward()
    optimizer.step()

## 模型推理

In [ ]:
# Predict mask tokens ans isNext
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(batch[0]))
print(text)
print()

print([number_dict[w.item()] for w in input_ids[0] if number_dict[w.item()] != '[PAD]'])
print()

logits_lm, logits_clsf = model(input_ids, segment_ids, masked_pos)
logits_lm = logits_lm.data.max(2)[1][0].data.numpy()
print('masked tokens list : ', [pos.item() for pos in masked_tokens[0] if pos.item() != 0])
print('predict masked tokens list : ', [pos for pos in logits_lm if pos != 0])
print()

logits_clsf = logits_clsf.data.max(1)[1].data.numpy()[0]
print('isNext : ', True if isNext else False)
print('predict isNext : ', True if logits_clsf else False)

# HF

## BERT

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Natural Language Processing
from textblob import TextBlob
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Model evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Utilities
from collections import Counter
import string
import warnings
warnings.filterwarnings("ignore")

# PyTorch and Transformers
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm

In [ ]:
train_df = pd.read_csv(
    '/kaggle/input/twitter-entity-sentiment-analysis/twitter_training.csv',
    header=None,names=['uid','entity','sentiment','content']
)

valid_df = pd.read_csv(
    '/kaggle/input/twitter-entity-sentiment-analysis/twitter_validation.csv',
    header=None,names=['uid','entity','sentiment','content']
)

train_df

In [ ]:
print(train_df.isnull().sum())

print(train_df.info())

print(train_df['sentiment'].unique())

In [ ]:
kv = {'Negative':0, 'Positive':1, 'Neutral':2, 'Irrelevant':3}
train_df['content'] = train_df['content'].astype(str)
train_df['sid'] = train_df['sentiment'].map(kv)
valid_df['content'] = valid_df['content'].astype(str)
valid_df['sid'] = valid_df['sentiment'].map(kv)
valid_df

### EDA

In [ ]:
sns.countplot(x='sentiment', data=train_df)
plt.title('Class Distribution - Positive vs Negative Reviews')
plt.show()

train_df['sentiment'].value_counts(normalize=True) * 100

In [ ]:
train_df['word_count'] = train_df['content'].apply(lambda x: len(x.split()))
train_df['char_count'] = train_df['content'].apply(lambda x: len(x))

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='word_count', hue='sentiment', multiple='stack', bins=50)
plt.title('Word Count Distribution by Sentiment')
plt.xlabel('Word Count')
plt.ylabel('Number of Reviews')
plt.show()

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='char_count', hue='sentiment', multiple='stack', bins=50)
plt.title('Character Count Distribution by Sentiment')
plt.xlabel('Character Count')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
word_count_sentiment = train_df.groupby('word_count')['sid'].mean()

plt.figure(figsize=(12, 6))
sns.lineplot(x=word_count_sentiment.index, y=word_count_sentiment.values)
plt.title('Sentiment Distribution Over Review Length (Word Count)')
plt.xlabel('Word Count')
plt.ylabel('Average Sentiment (1 = Positive, 0 = Negative)')
plt.show()

In [ ]:
def clean_text(text):
    text = text.lower()
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS and word not in string.punctuation]
    return words

positive_words = clean_text(''.join(train_df[train_df['sentiment'] == 'Positive']['content']))
negative_words = clean_text(''.join(train_df[train_df['sentiment'] == 'Negative']['content']))
neutral_words = clean_text(''.join(train_df[train_df['sentiment'] == 'Neutral']['content']))
irrelevant_words = clean_text(''.join(train_df[train_df['sentiment'] == 'Irrelevant']['content']))

positive_word_freq = Counter(positive_words)
most_common_positive = positive_word_freq.most_common(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in most_common_positive], y=[word[0] for word in most_common_positive])
plt.title('Top 20 Most Common Words in Positive Reviews')
plt.show()

negative_word_freq = Counter(negative_words)
most_common_negative = negative_word_freq.most_common(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in most_common_negative], y=[word[0] for word in most_common_negative])
plt.title('Top 20 Most Common Words in Negative Reviews')
plt.show()

neutral_words_freq = Counter(neutral_words)
most_common_neutral = neutral_words_freq.most_common(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in most_common_positive], y=[word[0] for word in most_common_neutral])
plt.title('Top 20 Most Common Words in Neutral Reviews')
plt.show()

irrelevant_word_freq = Counter(irrelevant_words)
most_common_irrelevant = irrelevant_word_freq.most_common(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in most_common_negative], y=[word[0] for word in most_common_irrelevant])
plt.title('Top 20 Most Common Words in Irrelevant Reviews')
plt.show()

In [ ]:
# Function to get n-grams
def get_top_n_grams(corpus, ngram_range=(2, 2), n=None):
    vec = CountVectorizer(ngram_range=ngram_range, stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

top_positive_bigrams = get_top_n_grams(train_df[train_df['sentiment'] == 'Positive']['content'], ngram_range=(2, 2), n=10)
top_negative_bigrams = get_top_n_grams(train_df[train_df['sentiment'] == 'Negative']['content'], ngram_range=(2, 2), n=10)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in top_positive_bigrams], y=[word[0] for word in top_positive_bigrams])
plt.title('Top 10 Bigrams in Positive Reviews')
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in top_negative_bigrams], y=[word[0] for word in top_negative_bigrams])
plt.title('Top 10 Bigrams in Negative Reviews')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(train_df[train_df['sentiment'] == 'Positive']['word_count'], color='green', bins=50, label='Positive', kde=True)
sns.histplot(train_df[train_df['sentiment'] == 'Negative']['word_count'], color='red', bins=50, label='Negative', kde=True)
plt.title('Word Count Distribution for Positive vs Negative Reviews')
plt.xlabel('Word Count')
plt.ylabel('Number of Reviews')
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
sns.histplot(train_df[train_df['sentiment'] == 'Positive']['char_count'], color='green', bins=50, label='Positive', kde=True)
sns.histplot(train_df[train_df['sentiment'] == 'Negative']['char_count'], color='red', bins=50, label='Negative', kde=True)
plt.title('Character Count Distribution for Positive vs Negative Reviews')
plt.xlabel('Character Count')
plt.ylabel('Number of Reviews')
plt.legend()
plt.show()

In [ ]:
positive_words_set = set(positive_words)
negative_words_set = set(negative_words)

common_words = positive_words_set.intersection(negative_words_set)

unique_positive_words = positive_words_set - common_words

unique_negative_words = negative_words_set - common_words

print(f"Number of common words between positive and negative reviews: {len(common_words)}")
print(f"Number of unique words in positive reviews: {len(unique_positive_words)}")
print(f"Number of unique words in negative reviews: {len(unique_negative_words)}")

In [ ]:
common_word_freq = Counter(common_words)
most_common_shared = common_word_freq.most_common(20)

plt.figure(figsize=(10, 6))
sns.barplot(x=[word[1] for word in most_common_shared], y=[word[0] for word in most_common_shared])
plt.title('Top 20 Most Common Words Shared Between Positive and Negative Reviews')
plt.show()

In [ ]:
def count_punctuation(text, punct):
    return text.count(punct)

train_df['exclamation_count'] = train_df['content'].apply(lambda x: count_punctuation(x, '!'))
train_df['question_count'] = train_df['content'].apply(lambda x: count_punctuation(x, '?'))
train_df['ellipsis_count'] = train_df['content'].apply(lambda x: count_punctuation(x, '...'))

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='exclamation_count', hue='sentiment', multiple='stack', bins=30)
plt.title('Exclamation Mark (!) Distribution by Sentiment')
plt.xlabel('Number of Exclamation Marks')
plt.ylabel('Number of Reviews')
plt.show()

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='question_count', hue='sentiment', multiple='stack', bins=30)
plt.title('Question Mark (?) Distribution by Sentiment')
plt.xlabel('Number of Question Marks')
plt.ylabel('Number of Reviews')
plt.show()

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='ellipsis_count', hue='sentiment', multiple='stack', bins=30)
plt.title('Ellipsis (...) Distribution by Sentiment')
plt.xlabel('Number of Ellipses')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
from wordcloud import WordCloud

positive_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(positive_words))

plt.figure(figsize=(10, 6))
plt.imshow(positive_wordcloud, interpolation='bilinear')
plt.title('Word Cloud for Positive Reviews')
plt.axis('off')
plt.show()

negative_wordcloud = WordCloud(width=800, height=400, background_color='black').generate(' '.join(negative_words))

plt.figure(figsize=(10, 6))
plt.imshow(negative_wordcloud, interpolation='bilinear')
plt.title('Word Cloud for Negative Reviews')
plt.axis('off')
plt.show()

In [ ]:
def get_polarity(text):
    return TextBlob(text).sentiment.polarity

def get_subjectivity(text):
    return TextBlob(text).sentiment.subjectivity

train_df['polarity'] = train_df['content'].apply(get_polarity)
train_df['subjectivity'] = train_df['content'].apply(get_subjectivity)

# Visualize polarity distribution
plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='polarity', hue='sentiment', multiple='stack', bins=50, kde=True)
plt.title('Polarity Distribution by Sentiment')
plt.xlabel('Polarity')
plt.ylabel('Number of Reviews')
plt.show()

# Visualize subjectivity distribution
plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='subjectivity', hue='sentiment', multiple='stack', bins=50, kde=True)
plt.title('Subjectivity Distribution by Sentiment')
plt.xlabel('Subjectivity')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
positive_reviews = train_df[train_df['sentiment'] == 'Positive']['content']
negative_reviews = train_df[train_df['sentiment'] == 'Negative']['content']

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

tfidf_positive = tfidf.fit_transform(positive_reviews)
tfidf_negative = tfidf.fit_transform(negative_reviews)

positive_top_words = pd.DataFrame(tfidf_positive.toarray(), columns=tfidf.get_feature_names_out()).mean().sort_values(ascending=False)[:20]
negative_top_words = pd.DataFrame(tfidf_negative.toarray(), columns=tfidf.get_feature_names_out()).mean().sort_values(ascending=False)[:20]

plt.figure(figsize=(10, 6))
sns.barplot(x=positive_top_words.values, y=positive_top_words.index)
plt.title('Top 20 TF-IDF Words in Positive Reviews')
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x=negative_top_words.values, y=negative_top_words.index)
plt.title('Top 20 TF-IDF Words in Negative Reviews')
plt.show()

### 模型

In [ ]:
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    train_df['content'].values, train_df['sid'].values, test_size=0.2, random_state=42
)

In [ ]:
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the data for BERT
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=512)
valid_encodings = tokenizer(list(valid_texts), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(list(valid_df['content'].values), truncation=True, padding=True, max_length=512)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(valid[idx]) for key, valid in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset = CustomDataset(val_encodings, valid_labels)
test_dataset = CustomDataset(test_encodings, valid_df['sid'].values)

In [ ]:
# Load the BERT model for sequence classification (binary classification)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

optimizer = AdamW(model.parameters(), lr=2e-5)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

### 训练

In [ ]:
def evaluate(model, valid_loader, device):
    model.eval()
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, preds)
    print(f'Validation Accuracy: {accuracy}')
    print(classification_report(true_labels, preds))

def train(model, train_loader, valid_loader, optimizer, device, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        print(f'Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}')
        evaluate(model, valid_loader, device)


train(model, train_loader, valid_loader, optimizer, device, epochs=3)

### 评估

In [ ]:
def evaluate_on_test(model, test_loader, device):
    model.eval()
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, preds)
    print(f'Test Accuracy: {accuracy}')
    print(classification_report(true_labels, preds))

evaluate_on_test(model, test_loader, device)

### 测试

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

def predict_sentiment(sentence):
    encoded_input = tokenizer.encode_plus(sentence, add_special_tokens=True, return_tensors='pt', truncation=True, padding=True, max_length=512)
    input_ids = encoded_input['input_ids'].to(device)
    attention_mask = encoded_input['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids, attention_mask=attention_mask)

    probabilities = torch.softmax(output.logits, dim=1)
    predicted_label = torch.argmax(probabilities, dim=1).cpu().numpy()[0]

    sentiments = {0: 'Negative', 1: 'Positive'}
    return sentiments[predicted_label], probabilities

user_sentence = input("Enter a sentence for sentiment analysis: ")

# Predict sentiment
predicted_sentiment, prob = predict_sentiment(user_sentence)

# Print the results
print(f"Predicted Sentiment: {predicted_sentiment}")
print(f"Probabilities: {prob}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

def predict_sentiment(sentence):
    encoded_input = tokenizer.encode_plus(sentence, add_special_tokens=True, return_tensors='pt', truncation=True, padding=True, max_length=512)
    input_ids = encoded_input['input_ids'].to(device)
    attention_mask = encoded_input['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids, attention_mask=attention_mask)

    probabilities = torch.softmax(output.logits, dim=1)
    predicted_label = torch.argmax(probabilities, dim=1).cpu().numpy()[0]

    sentiments = {0: 'Negative', 1: 'Positive'}
    return sentiments[predicted_label], probabilities

user_sentence = input("Enter a sentence for sentiment analysis: ")

# Predict sentiment
predicted_sentiment, prob = predict_sentiment(user_sentence)

# Print the results
print(f"Predicted Sentiment: {predicted_sentiment}")
print(f"Probabilities: {prob}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

def predict_sentiment(sentence):
    encoded_input = tokenizer.encode_plus(sentence, add_special_tokens=True, return_tensors='pt', truncation=True, padding=True, max_length=512)
    input_ids = encoded_input['input_ids'].to(device)
    attention_mask = encoded_input['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids, attention_mask=attention_mask)

    probabilities = torch.softmax(output.logits, dim=1)
    predicted_label = torch.argmax(probabilities, dim=1).cpu().numpy()[0]

    sentiments = {0: 'Negative', 1: 'Positive'}
    return sentiments[predicted_label], probabilities

user_sentence = input("Enter a sentence for sentiment analysis: ")

# Predict sentiment
predicted_sentiment, prob = predict_sentiment(user_sentence)

# Print the results
print(f"Predicted Sentiment: {predicted_sentiment}")
print(f"Probabilities: {prob}")

## ModernBERT

In [1]:
%%capture
!pip install -U transformers>=4.48.0 -q
!pip flash-attn -q

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
from pprint import pprint

2025-03-28 08:14:14.173424: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-28 08:14:14.173556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-28 08:14:14.303537: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
#model_id = "answerdotai/ModernBERT-base"
model_id = "/kaggle/input/modernbert/transformers/large/2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id, device_map='auto', torch_dtype=torch.bfloat16)
model

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=1024, out_features=3072, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=1024, out_features=1024, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=1024, out_features=5248, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=2624, out_features=1024, bias=False)
        )
      )
    

In [ ]:
text = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model(**inputs)

# To get predictions for the mask:

masked_index = inputs["input_ids"][0].tolist().index(tokenizer.mask_token_id)
predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)
print("Predicted token:", predicted_token)
# Predicted token:  Paris

In [ ]:
pipe = pipeline(
    "fill-mask",
    model=checkpoint,
    torch_dtype=torch.bfloat16,
)

input_text = "He walked to the [MASK]."
results = pipe(input_text)
pprint(results)

# Bert4Rec

In [ ]:
# Loading required libraries
import os
import numpy as np
import ast
import gc
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForMaskedLM, AdamW

In [ ]:
# Function to load the data and split it into training and testing sets
def load_and_split_data(file_path, test_size=0.2, random_state=42):
    data = pd.read_csv(file_path)
    train_data, test_data = train_test_split(data, test_size=test_size, random_state=random_state)
    return train_data.reset_index(drop=True), test_data.reset_index(drop=True)

# Function to convert the 'product_id-list' column to lists of integers from the object type
def convert_product_id_list_to_integers(data):
    data['product_id-list'] = data['product_id-list'].apply(ast.literal_eval)
    return data

# Function to convert the 'product_id-list' column to lists of strings from the list of integers type
def convert_product_id_list_to_string_lists(data):
    data['product_id-list'] = data['product_id-list'].apply(lambda x: [str(item) for item in x])

    return data

# Function to extract token lists from the 'product_id-list' column and flatten them
def extract_and_flatten_prod_id(data):
    token_lists = data['product_id-list'].tolist()
    flat_tokens = [token for sublist in token_lists for token in sublist]
    return flat_tokens

# Function to extract unique token IDs
def extract_unique_ids(col):
    return list(set(col))

# Function to convert product IDs from integers to strings
def convert_product_ids_to_strings(product_ids):
    return [str(product_id) for product_id in product_ids]

In [ ]:
# Example usage:
file_path = '/kaggle/input/e-commerce-product-purchase-data/final_data.csv'
train_data, test_data = load_and_split_data(file_path)
train_data = convert_product_id_list_to_integers(train_data)
test_data = convert_product_id_list_to_integers(test_data)
train_data = convert_product_id_list_to_string_lists(train_data)
test_data = convert_product_id_list_to_string_lists(test_data)
product_ids = extract_and_flatten_prod_id(train_data)
unique_product_ids = extract_unique_ids(product_ids)
unique_product_ids = convert_product_ids_to_strings(unique_product_ids)

In [ ]:
# Load the pretrained tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokens_to_replace = list(tokenizer.vocab.keys())
i=0
# Replace tokens with product IDs
for token in tokens_to_replace:
    if token not in tokenizer.all_special_tokens:
        token_id = tokenizer.vocab[token]
        del tokenizer.vocab[token]  # Remove the token from the vocabulary
        tokenizer.vocab[unique_product_ids[i]] = token_id  # Replace it with a product ID
        # Replace the token in the ids_to_tokens dictionary
        tokenizer.ids_to_tokens.pop(token_id)
        tokenizer.ids_to_tokens[token_id] = unique_product_ids[i]
        i=i+1
tokenizer.add_tokens(unique_product_ids[i:])

### Preparing Training dataset and loading the bert model

In [ ]:
# Tokenize the text data
tokenized_data = train_data['product_id-list'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))

# Define maximum sequence length
max_length = max(len(seq) for seq in tokenized_data)

# Pad tokenized sequences to maximum length
padded_sequences = np.array([seq + [tokenizer.pad_token_id] * (max_length - len(seq)) for seq in tokenized_data])

# Convert padded sequences to PyTorch tensors
labels = torch.tensor(padded_sequences)

# Define a function to apply masking
def mask_tokens(token_sequences):
    masked_sequence = []
    for tokens in token_sequences:
        # Find the indices of special tokens
        special_token_indices = [i for i, token_id in enumerate(tokens) if token_id in tokenizer.all_special_ids]
        # Determine the number of non-special tokens
        num_non_special_tokens = len(tokens) - len(special_token_indices)
        # Determine the number of tokens to mask, excluding special tokens
        num_tokens_to_mask = int(num_non_special_tokens * 0.5)
        # Randomly select non-special tokens to mask
        masked_indices = np.random.choice([i for i in range(len(tokens)) if i not in special_token_indices],
                                          num_tokens_to_mask, replace=False)
        # Mask the selected tokens
        for index in masked_indices:
            tokens[index] = tokenizer.mask_token_id
        masked_sequence.append(tokens)
    return masked_sequence

# Apply masking to the tokenized text
masked_sequences = mask_tokens(tokenized_data)

# Pad tokenized sequences to maximum length
masked_padded_sequences = np.array([seq + [tokenizer.pad_token_id] * (max_length - len(seq)) for seq in masked_sequences])

# Convert padded sequences to PyTorch tensors
input_ids = torch.tensor(masked_padded_sequences)

# Create attention masks
attention_masks = torch.where(input_ids != tokenizer.pad_token_id, torch.tensor(1), torch.tensor(0))

# Create labels_val
labels = torch.where(input_ids == tokenizer.mask_token_id, labels, torch.tensor(-100))

# Convert data into PyTorch Dataset
dataset = torch.utils.data.TensorDataset(input_ids, attention_masks, labels)

# Define DataLoader with reduced batch size
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)

# Define BERT model with reduced embedding layer
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# Modify model architecture to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Define optimizer with reduced learning rate
optimizer = AdamW(model.parameters(), lr=0.0001)

# Training loop with gradient accumulation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

### Preparing Validation dataset

In [ ]:
# Tokenize the text data
tokenized_data_val = test_data['product_id-list'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))

# Pad tokenized sequences to maximum length
padded_sequences_val = np.array([seq + [tokenizer.pad_token_id] * (max_length - len(seq)) for seq in tokenized_data_val])

# Convert padded sequences to PyTorch tensors
labels_val = torch.tensor(padded_sequences_val)

# Apply masking to the tokenized text
masked_sequences_val = mask_tokens(tokenized_data_val)

# Pad tokenized sequences to maximum length
masked_padded_sequences_val = np.array([seq + [tokenizer.pad_token_id] * (max_length - len(seq)) for seq in masked_sequences_val])

# Convert padded sequences to PyTorch tensors
input_ids_val = torch.tensor(masked_padded_sequences_val)

# Create labels_val
labels_val = torch.where(input_ids_val == tokenizer.mask_token_id, labels_val, torch.tensor(-100))

# Create attention masks
attention_masks_val = torch.where(input_ids_val != tokenizer.pad_token_id, torch.tensor(1), torch.tensor(0))

# Convert data into PyTorch Dataset
dataset_val = torch.utils.data.TensorDataset(input_ids_val, attention_masks_val, labels_val)

# Define DataLoader with reduced batch size
valid_loader = DataLoader(dataset_val, batch_size=256, shuffle=True)

In [ ]:
accumulation_steps = 256
num_epochs = 2
train_losses = []
valid_losses = []


for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0
    
    # Training loop
    for index, batch in enumerate(train_loader):
        optimizer.zero_grad()
        batch = tuple(t.to(device) for t in batch)
        inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'labels': batch[2]}
        labels = inputs['labels']
        outputs = model(inputs['input_ids'], attention_mask=inputs['attention_mask'], labels=labels)
        train_loss = outputs.loss
        total_train_loss += train_loss.item()

        # Backpropagation and gradient accumulation
        train_loss = train_loss / accumulation_steps
        train_loss.backward()
        if (index + 1) % accumulation_steps == 0 or index == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation loop
    model.eval()
    total_val_loss = 0.0

    for index, batch in enumerate(valid_loader):
        with torch.no_grad():  # Disable gradient computation during validation
            batch = tuple(t.to(device) for t in batch)
            inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'labels': batch[2]}
            labels = inputs['labels']
            outputs = model(inputs['input_ids'], attention_mask=inputs['attention_mask'], labels=labels)
            val_loss = outputs.loss
            total_val_loss += val_loss.item()

    # Calculate average validation loss for the epoch
    avg_val_loss = total_val_loss / len(valid_loader)
    
    # Append losses to lists
    train_losses.append(avg_train_loss)
    valid_losses.append(avg_val_loss)
    
    print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}')

#### Loss graph

In [ ]:
from matplotlib import pyplot as plt

# Plot the training and validation losses
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
plt.plot(range(1, len(valid_losses) + 1), valid_losses, label='Validation Loss')

# Add legend
plt.legend()

# Set x-axis limit and label
plt.xlim(1, len(train_losses))
plt.xlabel('Epoch')

# Show plot
plt.show()

In [ ]:
# save
output_model = './model.pth'
def save(model, optimizer):
    # save
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, output_model)

save(model, optimizer)

In [ ]:
tokenizer.save_pretrained('./tokenizer.pth')

In [ ]:
model.save_pretrained('./model')

In [ ]:
# load
output_model = '/kaggle/input/bert4rec-training/model.pth'
checkpoint = torch.load(output_model, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
j=111
i= len(test_data['product_id-list'][j])
print(test_data['product_id-list'][j])
pred_tokens=tokenizer.encode(test_data['product_id-list'][j], add_special_tokens=True)

# Set the masked token ID
pred_tokens[i] = tokenizer.mask_token_id

# Convert input to tensor
input_tensor = torch.tensor(pred_tokens).unsqueeze(0).to(device)  # Add batch dimension

# Make prediction
with torch.no_grad():
    outputs = model(input_tensor)

# Get prediction logits
logits = outputs.logits

# Get predicted token ID
predicted_token_id = torch.argmax(logits[0,i]).item()

# Convert predicted token ID to token
predicted_token = tokenizer.decode(predicted_token_id)

print("Predicted Token:", predicted_token)